<a href="https://colab.research.google.com/github/Rathinagrace/nhg-clinical-informatics-project/blob/main/Milestone_1_Data_Generation_And_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Project Title: Data-Driven Optimization of EMR Inpatient Discharge Workflows
**Institution Context:** National Healthcare Group (NHG) — Next Generation Electronic Medical Record (NGEMR) Initiative  
**Author:** Rathina Grace Monica Abraham Geoffrey  
**Domain Focus:** Clinical Analytics, Workflow Redesign, Decision Support, Master Data Management (MDM)

---

## 📖 1. Project Introduction
In modern tertiary hospital groups like the National Healthcare Group (NHG), maximizing bed turnover efficiency is essential to reducing Emergency Department boarding times and optimizing resource allocation. When a physician decides a patient is medically fit for discharge, a complex operational chain is triggered within the Electronic Medical Record (EMR) system. This involves multidisciplinary collaboration across physicians, ward nurses, pharmacists, and medical transport coordinators.

This project demonstrates the application of clinical informatics principles to bridge raw electronic health data with frontline operational improvements. By evaluating simulated clinical event logs, this framework isolates administrative inefficiencies, identifies clinical documentation bottlenecks, and translates raw metrics into actionable decision support interventions for hospital leadership.

## 🎯 2. Core Project Objectives

The primary objective of this project is to analyze clinical data structures to optimize hospital operations. Specifically, the workflow aims to:
1. **Model an Epic Clarity Database Environment:** Establish a relational framework utilizing standard medical encounter metrics to evaluate administrative latency.
2. **Execute Clinical Programmatic Data Cleaning:** Implement data profiling techniques to resolve missing fields, unformatted text strings, and timestamp anomalies injected during front-end EMR clinical charting.
3. **Isolate Workflow Bottlenecks via Descriptive Analytics:** Programmatically measure "Discharge Latency"—the elapsed time between a physician signing a discharge order and the patient’s physical departure from the ward.
4. **Deliver Executive Decision Support:** Synthesize operational Key Performance Indicators (KPIs) into an enterprise-grade interactive dashboard mapping workflow bottlenecks by clinical department.

## 📐 3. Data Architecture: The Dimensional Star Schema

To optimize large enterprise healthcare data reporting platforms (such as **Epic Caboodle**), raw data is structured into a **Star Schema**. This framework consists of a central **Fact Table** containing measurable quantitative events, linked directly to surrounding descriptive **Dimension Tables**.

### 🗺️ System Relationship Map:
* **[PATIENT_DIM]** (Patient Demographics) ───via `PAT_ID`───┐
* **[ORDER_PROC_DIM]** (Physician Orders) ───via `PAT_ENC_CSN_ID`───┼───► **[★ PAT_ENC_HSP_FACT]**
* **[IP_FLWSHT_MEAS_DIM]** (Nursing Checklists) ──via `PAT_ENC_CSN_ID`───┘

---

### 📑 Database Table Components:

1.  **★ PAT_ENC_HSP_FACT (Central Fact Table):** Tracks the primary operational event—the hospital stay. Every admission generates a unique transactional key called a *Contact Serial Number* (`PAT_ENC_CSN_ID`). It links the patient record, operational ward IDs, and physical tracking timestamps.
2.  **PATIENT_DIM (Dimension Table):** Stores slowly changing master demographic details including patient metadata, name strings, and system categorical identifiers.
3.  **ORDER_PROC_DIM (Dimension Table):** Captures transaction logs of orders signed by physicians, documenting exactly when the medical clearance sequence was initiated.
4.  **IP_FLWSHT_MEAS_DIM (Dimension Table):** Records discrete clinical charting flowsheets managed by ward nursing staff, tracking operational milestones such as transport readiness and medication reconciliations.

### 📑 Database Components:
1. **★ PAT_ENC_HSP_FACT (Central Fact Table):** Tracks the primary operational event—the hospital stay. Every admission generates a unique key called a *Contact Serial Number* (`PAT_ENC_CSN_ID`). It links the patient record, timestamps, and physical movement tracking.
2. **PATIENT_DIM (Dimension Table):** Stores slowly changing master demographic details including patient metadata and system categorical identifiers.
3. **ORDER_PROC_DIM (Dimension Table):** Captures transaction logs of orders signed by physicians, documenting exactly when the medical clearance sequence was initiated.
4. **IP_FLWSHT_MEAS_DIM (Dimension Table):** Records discrete clinical charting flowsheets managed by nursing staff, tracking operational checklists such as transport readiness and medication reconciliations.

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

num_patients = 5000

print("⏳ Initiating healthcare data simulation engine (5,000 records)...")

# 1. GENERATE PATIENT DIMENSION
patient_ids = [f"PAT_{10000 + i}" for i in range(num_patients)]
names = [f"Patient, Sample {i}" for i in range(num_patients)]
start_date = datetime(1945, 1, 1)
birth_days = [start_date + timedelta(days=int(random.randint(0, 28000))) for _ in range(num_patients)]
sex_codes = [random.choice([1, 2]) for _ in range(num_patients)]

df_patient = pd.DataFrame({
    'PAT_ID': patient_ids,
    'PAT_NAME': names,
    'BIRTH_DATE': [b.strftime('%Y-%m-%d') for b in birth_days],
    'SEX_C': sex_codes
})

# 2. GENERATE HOSPITAL ENCOUNTERS, ORDERS & FLOWSHEETS
encounters = []
orders = []
flowsheets = []

# Wards: 101 = General Medicine, 102 = Cardiology, 103 = Orthopaedics
depts = [101, 102, 103]
csn_start = 7000001
order_start = 4000001
flow_start = 8000001
base_admission_date = datetime(2026, 1, 1)

for i in range(num_patients):
    csn = csn_start + i
    pat_id = patient_ids[i]
    dept = random.choice(depts)

    los_days = random.randint(2, 10)
    admit_time = base_admission_date + timedelta(days=random.randint(0, 150), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    order_time = admit_time + timedelta(days=los_days, hours=random.randint(0, 2))

    # Injecting the Structural Operational Bottleneck (Orthopaedics Dept 103)
    if dept == 103:
        latency_hours = random.uniform(5.5, 9.5)
        # Injecting Malformed Strings for the data cleaning exercise
        transport_status = random.choice([" delayed ", "  DELAYED  ", " Delayed"])
    else:
        latency_hours = random.uniform(1.2, 2.8)
        transport_status = random.choice(["Completed", "  completed ", "COMPLETED"])

    disch_time = order_time + timedelta(hours=latency_hours)
    disch_disp = random.choice([1, 1, 1, 1, 2, 3]) # 1=Home, 2=Rehab, 3=Nursing Home

    # Anomaly Type 1: Missing Data (Simulating aborted/canceled encounters with missing discharge times)
    if random.random() < 0.03:
        disch_time = np.nan
        disch_disp = np.nan

    encounters.append([csn, pat_id, admit_time, disch_time, disch_disp, dept])

    # Generate Doctor Orders (PROC_ID 9001 = Discharge Order)
    order_id = order_start + i
    orders.append([order_id, csn, 9001, order_time, f"DOC_{random.randint(100, 250)}"])

    # Generate Flowsheet Checklist Items
    # 4001 = Medication Reconciliation, 4002 = Transport Coordination
    med_recon_time = order_time + timedelta(minutes=random.randint(10, 50))
    transport_time = order_time + timedelta(minutes=random.randint(20, 140))

    # Anomaly Type 2: Corrupted Timestamps (Missing recording timestamps in flowsheet charting)
    if random.random() < 0.02:
        transport_time = np.nan

    flowsheets.append([flow_start + (i*2), csn, 4001, random.choice(["Completed", "COMPLETED"]), med_recon_time])
    flowsheets.append([flow_start + (i*2) + 1, csn, 4002, transport_status, transport_time])

# Build DataFrames
df_enc = pd.DataFrame(encounters, columns=['PAT_ENC_CSN_ID', 'PAT_ID', 'HOSP_ADMSN_TIME', 'HOSP_DISCH_TIME', 'DISCH_DISP_C', 'ADMIT_DEPT_ID'])
df_ord = pd.DataFrame(orders, columns=['ORDER_ID', 'PAT_ENC_CSN_ID', 'PROC_ID', 'ORDERING_DATE', 'AUTHRZING_PROV_ID'])
df_flow = pd.DataFrame(flowsheets, columns=['FSD_ID', 'PAT_ENC_CSN_ID', 'FLW_MEAS_ID', 'MEAS_VALUE', 'RECORDING_TIME'])

# Export DataFrames to local environment CSVs
df_patient.to_csv('patient_dim.csv', index=False)
df_enc.to_csv('pat_enc_hsp_fact.csv', index=False)
df_ord.to_csv('order_proc_dim.csv', index=False)
df_flow.to_csv('ip_flwsht_meas_dim.csv', index=False)

print("🎉 Success! Generated 4 files containing 5,000 simulated records populated with workflow anomalies.")

⏳ Initiating healthcare data simulation engine (5,000 records)...
🎉 Success! Generated 4 files containing 5,000 simulated records populated with workflow anomalies.
